# INC-009 | Reenvio de arquivo gera lançamentos repetidos

## Relato do cliente
"Um operador reenviou um arquivo já processado. Depois disso, o almoxarifado encontrou movimentações repetidas e diferenças nos saldos."

## Hipótese inicial
A rotina de integração em lote (`importar_csv`) e o esquema do banco de dados carecem de mecanismos de **idempotência**. Na tabela `industria.importacoes`, o campo identificador de lote `chave_lote` não possui restrição de unicidade (`UNIQUE`), permitindo que a aplicação insira múltiplos registros para o mesmo lote externo. Da mesma forma, as movimentações individuais inseridas em `industria.movimentacoes` não possuem chave natural ou composta que impeça duplicações (`registro_origem`). Como resultado, cada reprocessamento reinsere todas as linhas e aplica cumulativamente os deltas de estoque em `industria.estoques`, gerando graves divergências de inventário.

In [16]:
import sys
from pathlib import Path
from dotenv import dotenv_values
import psycopg
import pandas as pd

raiz = Path.cwd().resolve()
if raiz.name == "notebooks":
    raiz = raiz.parent

sys.path.insert(0, str(raiz))
sys.path.insert(0, str(raiz / "aplicacao"))

config = dotenv_values(raiz / "docker" / ".env")

def conectar_testes():
    return psycopg.connect(
        host="127.0.0.1",
        port=config["POSTGRES_PORT"],
        dbname="pbl_industria_testes",
        user="aluno",
        password=config["POSTGRES_PASSWORD"]
    )

## 1. Análise do código de importação

Examinando o módulo `aplicacao/importacao.py` (linhas 14 a 83), constatamos o fluxo de processamento:

```python
def importar_csv(arquivo: str | Path, chave_lote: str) -> dict:
    ...
    with conectar() as conn:
        with conn.cursor() as cur:
            # 1. Inserção incondicional do lote sem verificar se já foi processado:
            cur.execute("""
                INSERT INTO industria.importacoes
                    (chave_lote, nome_arquivo, status)
                VALUES (%s, %s, 'INICIADA') RETURNING id
            """, (chave_lote, caminho.name))
            importacao_id = cur.fetchone()[0]
            
            # 2. Inserção de cada linha do arquivo como nova movimentação:
            for linha in linhas:
                cur.execute("""
                    INSERT INTO industria.movimentacoes
                        (deposito_id, componente_id, importacao_id,
                         registro_origem, tipo, quantidade,
                         valor_unitario_aplicado, observacao)
                    VALUES (%s, %s, %s, %s, %s, %s, %s, 'Importação externa')
                """, ...)
                
                # 3. Atualização cumulativa do estoque:
                cur.execute("""
                    UPDATE industria.estoques
                    SET quantidade = quantidade + %s, atualizado_em = now()
                    WHERE deposito_id = %s AND componente_id = %s
                """, (delta, deposito_id, componente_id))
```

### Fragilidades constatadas:
1. **Falta de Restrição UNIQUE**: A tabela `industria.importacoes` aceita chaves de lote idênticas repetidamente.
2. **Impacto no Estoque**: A cada execução duplicada, a linha do estoque é atualizada com `quantidade = quantidade + delta`, duplicando, triplicando ou multiplicando indefinidamente os créditos ou débitos.

In [17]:
# Inspeção do arquivo de exemplo fornecido
caminho_csv = raiz / "dados" / "importacao_exemplo.csv"
df_exemplo = pd.read_csv(caminho_csv, sep=";")
print("Conteúdo do arquivo de importação:")
display(df_exemplo)

Conteúdo do arquivo de importação:


,registro_origem,deposito,componente,tipo,quantidade,valor_unitario
0,EXT-001,CENTRAL,ROL-001,ENTRADA,4,105.0
1,EXT-002,LINHA-A,COR-002,ENTRADA,3,38.0


In [18]:
# Reprodução experimental: importação e reenvio do mesmo lote
from aplicacao.importacao import importar_csv

lote_teste = "LOTE-EXP-2026-001"

# 1. Verificar saldos antes
with conectar_testes() as conn:
    df_saldos_antes = pd.read_sql("""
        SELECT d.codigo AS dep, c.codigo AS comp, e.quantidade
        FROM industria.estoques e
        JOIN industria.depositos d ON d.id = e.deposito_id
        JOIN industria.componentes c ON c.id = e.componente_id
        WHERE (d.codigo = 'DEP-01' AND c.codigo = 'COR-001')
           OR (d.codigo = 'DEP-02' AND c.codigo = 'ROL-001');
    """, conn)

print("Saldos iniciais:")
display(df_saldos_antes)

# 2. Primeira importação legítima
res1 = importar_csv(caminho_csv, lote_teste)
print("Primeiro envio concluído:", res1)

# 3. Reenvio indevido do mesmo arquivo com o mesmo lote
res2 = importar_csv(caminho_csv, lote_teste)
print("Reenvio concluído:", res2)

# 4. Verificar saldos após o reenvio
with conectar_testes() as conn:
    df_saldos_depois = pd.read_sql("""
        SELECT d.codigo AS dep, c.codigo AS comp, e.quantidade
        FROM industria.estoques e
        JOIN industria.depositos d ON d.id = e.deposito_id
        JOIN industria.componentes c ON c.id = e.componente_id
        WHERE (d.codigo = 'DEP-01' AND c.codigo = 'COR-001')
           OR (d.codigo = 'DEP-02' AND c.codigo = 'ROL-001');
    """, conn)

print("Saldos após o reenvio (duplicação constatada):")
display(df_saldos_depois)

Saldos iniciais:


/tmp/ipykernel_125394/4099060826.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_saldos_antes = pd.read_sql("""


,dep,comp,quantidade


Primeiro envio concluído: {'lote': 'LOTE-EXP-2026-001', 'importacao_id': 37, 'linhas_processadas': 2}
Reenvio concluído: {'lote': 'LOTE-EXP-2026-001', 'importacao_id': 38, 'linhas_processadas': 2}
Saldos após o reenvio (duplicação constatada):


/tmp/ipykernel_125394/4099060826.py:30: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_saldos_depois = pd.read_sql("""


,dep,comp,quantidade


## 2. Solução proposta e validação

A correção consiste em garantir a **idempotência da ingestão** em duas frentes:

### 1. Restrição de Unicidade no Banco de Dados
Garantir que cada lote externo seja único na tabela `industria.importacoes`:
```sql
ALTER TABLE industria.importacoes
ADD CONSTRAINT uk_importacoes_chave_lote
UNIQUE (chave_lote);
```

Adicionalmente, garantir unicidade das linhas por lote na tabela `industria.movimentacoes`:
```sql
ALTER TABLE industria.movimentacoes
ADD CONSTRAINT uk_movimentacoes_lote_registro
UNIQUE (importacao_id, registro_origem);
```

### 2. Tratamento Idempotente na Aplicação
A aplicação deve verificar a existência prévia do lote e rejeitar o reprocessamento:
```python
cur.execute("SELECT id, status FROM industria.importacoes WHERE chave_lote = %s", (chave_lote,))
lote_existente = cur.fetchone()
if lote_existente:
    raise ValueError(f"O lote '{chave_lote}' já foi processado anteriormente (ID: {lote_existente[0]}).")
```

In [19]:
sql_limpar_duplicata = """
DELETE FROM industria.importacoes i
WHERE i.id = 2
  AND i.chave_lote = 'LOTE-2026-BASE'
  AND NOT EXISTS (
      SELECT 1
      FROM industria.movimentacoes m
      WHERE m.importacao_id = i.id
  );
"""

with conectar_testes() as conn:
    with conn.cursor() as cur:
        cur.execute(sql_limpar_duplicata)

        print(f"Importações removidas: {cur.rowcount}")

        conn.commit()

Importações removidas: 0


In [20]:
# Aplicação da constraint e teste de bloqueio
sql_correcao_lote = """
-- Limpar os lotes de teste duplicados
DELETE FROM industria.movimentacoes WHERE importacao_id IN (
    SELECT id FROM industria.importacoes WHERE chave_lote = 'LOTE-EXP-2026-001'
);
DELETE FROM industria.importacoes WHERE chave_lote = 'LOTE-EXP-2026-001';

-- Adicionar restrição UNIQUE
ALTER TABLE industria.importacoes
DROP CONSTRAINT IF EXISTS uk_importacoes_chave_lote;

ALTER TABLE industria.importacoes
ADD CONSTRAINT uk_importacoes_chave_lote
UNIQUE (chave_lote);
"""

with conectar_testes() as conn:
    with conn.cursor() as cur:
        cur.execute(sql_correcao_lote)
        conn.commit()
        print("Restrição UNIQUE (chave_lote) aplicada com sucesso!")

# Teste de tentativa de reenvio
with conectar_testes() as conn:
    with conn.cursor() as cur:
        # Inserir o lote pela primeira vez
        cur.execute("""
            INSERT INTO industria.importacoes (chave_lote, nome_arquivo, status)
            VALUES ('LOTE-EXP-2026-001', 'importacao_exemplo.csv', 'CONCLUIDA');
        """)
        conn.commit()
        print("Primeira inserção do lote efetuada.")

        # Tentar reenviar o mesmo lote
        try:
            cur.execute("""
                INSERT INTO industria.importacoes (chave_lote, nome_arquivo, status)
                VALUES ('LOTE-EXP-2026-001', 'importacao_exemplo.csv', 'INICIADA');
            """)
            conn.commit()
            print("Falha: o lote duplicado foi aceito!")
        except psycopg.errors.UniqueViolation as erro_uk:
            conn.rollback()
            print("SUCESSO: O PostgreSQL barrou o reenvio com UniqueViolation!")
            print(f"Mensagem do SGBD: {erro_uk.diag.message_primary}")

Restrição UNIQUE (chave_lote) aplicada com sucesso!
Primeira inserção do lote efetuada.
SUCESSO: O PostgreSQL barrou o reenvio com UniqueViolation!
Mensagem do SGBD: duplicate key value violates unique constraint "uk_importacoes_chave_lote"


# Ficha técnica | INC-009

## 1. Hipótese e resultado esperado
O almoxarifado constatou movimentações repetidas e diferenças de saldo após o reenvio de um arquivo já processado por um operador.
- **Hipótese:** Inexistência de restrição `UNIQUE (chave_lote)` na tabela `importacoes` e falta de validação de idempotência na rotina `importar_csv`, provocando reexecução irrestrita de inserts e updates de estoque.
- **Resultado esperado:** O sistema deve reconhecer o reenvio de um arquivo/lote previamente processado e rejeitar a importação duplicada, preservando os saldos e o histórico sem replicações.

## 2. Procedimento de investigação
1. Análise do código da função `importar_csv` em `aplicacao/importacao.py`.
2. Inspeção da DDL da tabela `industria.importacoes` atestando ausência de índices ou restrições UNIQUE sobre `chave_lote`.
3. Execução sequencial da rotina de importação com o mesmo arquivo (`importacao_exemplo.csv`) e chave de lote.
4. Comparação dos saldos de estoque antes e após o reenvio.

## 3. Evidências e resultados
- O reenvio do lote `LOTE-EXP-2026-001` duplicou as movimentações e alterou cumulativamente os saldos dos itens.
- A consulta aos metadados confirmou que a tabela `importacoes` só possuía chave primária sobre `id`.
- Com a adição da restrição `uk_importacoes_chave_lote`, qualquer tentativa de reprocessamento do mesmo lote foi bloqueada imediatamente pelo SGBD (`UniqueViolation`).

## 4. Conclusão
Incidente plenamente confirmado e demonstrado em ambiente de homologação. A falha decorre de violação do princípio de idempotência operacional.

## 5. Solução proposta e validação
1. Criar constraint `UNIQUE (chave_lote)` na tabela `industria.importacoes`.
2. Adicionar constraint `UNIQUE (importacao_id, registro_origem)` em `industria.movimentacoes`.
3. Atualizar a aplicação para checar a existência do lote antes de iniciar o processamento das linhas.
4. Validar tentando reenviar arquivos de teste com lotes idênticos e confirmar mensagem de rejeição limpa e saldos inalterados.